# Project Manager Agent - API Evaluation Client

This notebook evaluates the running Project Manager API (`/api/v1/project/chat`) using a **Global Comprehensive Rubric**.

**Pre-requisites:**
- Backend API running on `http://localhost:8001`
- Valid Authentication Token

In [1]:
import requests
import json
import uuid
import pandas as pd
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Load env for Judge LLM API Key
load_dotenv(os.path.join(os.getcwd(), '..', '..', '.env'))

print("Imports ready.")

Imports ready.


In [2]:
# ==============================================================================
# CONFIGURATION & GLOBAL RUBRIC
# ==============================================================================

API_URL = "http://localhost:8001/api/v1/project/chat"
TEST_TOKEN = os.getenv("API_BEARER_TOKEN", "YOUR_BEARER_TOKEN_HERE")

if TEST_TOKEN == "YOUR_BEARER_TOKEN_HERE":
    print("⚠️ WARNING: No token set. API calls will likely fail with 401.")
else:
    print(f"✅ Token loaded (Length: {len(TEST_TOKEN)}) chrs")

HEADERS = {"Authorization": f"Bearer {TEST_TOKEN}", "Content-Type": "application/json"}

# --- UNIFIED GLOBAL RUBRIC ---
GLOBAL_RUBRIC = """
1. **ACCURACY & COMPLETENESS**:
   - Does the response directly answer the User's core question?
   - Does it include all necessary details (names, dates, statuses) specified in the Reference?
   - Is the information logically consistent (no obvious hallucinations)?

2. **TONE & PROFESSIONALISM**:
   - Is the tone helpful, polite, and appropriate for a Project Manager Assistant?
   - Is the language natural Vietnamese (no awkward machine translations)?

3. **DATA PRIVACY & FORMATTING**:
   - **CRITICAL**: Does it HIDE all raw technical IDs (UUIDs/Database IDs)?
   - Is the formatting clear (uses Bullet points, Bold text for emphasis)?
   - Are dates formatted in a readable way (e.g., DD/MM/YYYY)?
"""

✅ Token loaded (Length: 165) chrs


In [3]:
# ==============================================================================
# JUDGE LLM SETUP (Local)
# ==============================================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage


judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0.0,
    google_api_key=os.getenv('GEMINI_API_KEY')
)

e:\Individual\Projects\ProMeet\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [4]:
# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def call_api(query: str, thread_id: str) -> str:
    """Send request to backend API"""
    payload = {"query": query, "thread_id": thread_id}
    try:
        response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=60)
        response.raise_for_status()
        return response.json().get("response", "Error: No response key")
    except requests.exceptions.RequestException as e:
        print(f"HTTP Request Failed: {e}")
        return f"API_ERROR: {str(e)}"

def evaluate_response(query: str, agent_response: str, reference: str) -> Dict[str, Any]:
    """Evaluate response using the GLOBAL RUBRIC in System Prompt"""
    if not judge_llm:
        return {"score": 0, "evaluation": {}, "explanation": "No Judge LLM"}
    
    # SYSTEM PROMPT: The Law
    system_prompt = f"""Role: Strict QA Auditor.
    
    YOU MUST EVALUATE THE AGENT RESPONSE AGAINST THIS GLOBAL RUBRIC:
    {GLOBAL_RUBRIC}
    
    OUTPUT JSON ONLY:
    {{
        "criteria_scores": {{
            "accuracy_completeness": {{ "passed": true/false, "reason": "..." }},
            "tone_professionalism": {{ "passed": true/false, "reason": "..." }},
            "privacy_formatting": {{ "passed": true/false, "reason": "..." }}
        }},
        "final_score_10": <integer 0-10>,
        "summary": "<short executive summary>"
    }}
    """
    
    # USER MESSAGE: The Case
    user_content = f"""CASE DETAILS:
    User Query: "{query}"
    Reference (Expected Answer Intent): "{reference}"
    
    ACTUAL AGENT RESPONSE:
    "{agent_response}"
    """
    
    try:
        res = judge_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_content)])
        content = res.content.strip().replace('```json', '').replace('```', '')
        return json.loads(content)
    except Exception as e:
        return {"score": 0, "evaluation": {}, "explanation": f"Judge Error: {e}"}

In [5]:
REASONING_TEST_CASES = [
    {
        "id": 1,
        "query": "Trong dự án Agentic application, ai đang làm task sửa lỗi crash app khi upload ảnh, đã xong chưa? Nếu chưa thì hãy đưa tôi email của người đó",  
        "reference": "Must identify the specific task, state its status (Done/In Progress), and provide the assignee's email ONLY if not done."
    },
    {
        "id": 3,
        "query": "Trong dự án Agentic application, chuyển task của Tuấn từ to do sang review",
        "reference": "Must update the task's status to 'Review' and assign it to 'Tuấn'."
    },
    {
        "id": 4,
        "query": "Task nào trong dự án Agentic application có deadline trước 7/12?",
        "reference": "Must apply a Date Filter (< Dec 7th) and only list tasks matching that criteria."
    }
]

print(f"Loaded {len(REASONING_TEST_CASES)} test cases.")

Loaded 3 test cases.


In [6]:
results = []

print("🚀 Starting GLOBAL RUBRIC Evaluation...\n")

for case in REASONING_TEST_CASES:
    print(f"Processing Case {case['id']}...")
    
    # 1. Call API
    thread_id = str(uuid.uuid4())
    response_text = call_api(case['query'], thread_id)
    
    # 2. Evaluate using Global Rubric
    eval_result = evaluate_response(case['query'], response_text, case['reference'])
    
    results.append({
        "id": case['id'],
        "query": case['query'],
        "api_response": response_text,
        "final_score": eval_result.get("final_score_10", 0),
        "criteria_scores": eval_result.get("criteria_scores", {}),
        "summary": eval_result.get("summary", "Error")
    })
    
print("\n✅ Evaluation Complete.")

🚀 Starting GLOBAL RUBRIC Evaluation...

Processing Case 1...


Processing Case 3...
Processing Case 4...

✅ Evaluation Complete.


In [7]:
# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================

df = pd.DataFrame(results)

def format_criteria_scores(scores):
    if not isinstance(scores, dict): return "N/A"
    html = "<div style='text-align:left; font-size:0.9em'>"
    
    # Mapping keys to readable labels
    labels = {
        "accuracy_completeness": "Accuracy",
        "tone_professionalism": "Tone",
        "privacy_formatting": "Privacy/Fmt"
    }
    
    for key, val in scores.items():
        icon = "✅" if val.get("passed") else "❌"
        color = "green" if val.get("passed") else "red"
        display_key = labels.get(key, key)
        html += f"<div style='margin-bottom:2px'><span style='color:{color}'>{icon} <b>{display_key}</b></span>: {val.get('reason', '')}</div>"
    html += "</div>"
    return html

if not df.empty:
    display_df = df.copy()
    display_df['Criteria Breakdown'] = display_df['criteria_scores'].apply(format_criteria_scores)
    
    # Rename columns
    display_df = display_df[["id", "query", "api_response", "final_score", "Criteria Breakdown"]].rename(columns={
        "id": "ID",
        "query": "Query",
        "api_response": "Agent Response",
        "final_score": "Score /10"
    })

    # Styling
    styled_df = display_df.style\
        .map(lambda v: f'color: {"green" if v >= 8 else "red"}; font-weight: bold; font-size: 1.1em', subset=['Score /10'])\
        .set_properties(subset=['Query', 'Agent Response'], **{'text-align': 'left', 'white-space': 'pre-wrap', 'min-width': '200px'})\
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('vertical-align', 'top')]}
        ])\
        .hide()

    display(styled_df)
else:
    print("No results to display.")

ID,Query,Agent Response,Score /10,Criteria Breakdown
1,"Trong dự án Agentic application, ai đang làm task sửa lỗi crash app khi upload ảnh, đã xong chưa? Nếu chưa thì hãy đưa tôi email của người đó","Trong dự án ""Agentic application"", task ""Fix lỗi crash app khi upload ảnh quá 5MB"" đang được giao cho bạn Nam và task này vẫn chưa xong (trạng thái ""In Progress""). Email của bạn Nam là zzvandatzz9@gmail.com.",10,"✅ Accuracy: Phản hồi đã trả lời chính xác và đầy đủ tất cả các phần trong câu hỏi của người dùng: xác định đúng người thực hiện (Nam), trạng thái công việc (chưa xong), và cung cấp email theo đúng điều kiện yêu cầu.✅ Tone: Giọng văn chuyên nghiệp, hữu ích và sử dụng ngôn ngữ tiếng Việt tự nhiên, phù hợp với vai trò trợ lý quản lý dự án.✅ Privacy/Fmt: Phản hồi không chứa ID kỹ thuật nào. Mặc dù không sử dụng gạch đầu dòng, câu văn ngắn gọn và rõ ràng, không cần định dạng phức tạp."
3,"Trong dự án Agentic application, chuyển task của Tuấn từ to do sang review","OK. Task ""Tích hợp cổng thanh toán VNPay"" của Tuấn đã được chuyển sang trạng thái Review.",10,"✅ Accuracy: Phản hồi xác nhận chính xác hành động được yêu cầu: chuyển trạng thái task của Tuấn sang Review. Mặc dù nó giả định tên task là 'Tích hợp cổng thanh toán VNPay', điều này có thể chấp nhận được trong một bối cảnh thực tế để làm rõ và xác nhận.✅ Tone: Giọng văn chuyên nghiệp, trực tiếp và hữu ích. Ngôn ngữ tiếng Việt tự nhiên.✅ Privacy/Fmt: Phản hồi không chứa ID kỹ thuật và được trình bày rõ ràng dưới dạng một câu đơn giản."
4,Task nào trong dự án Agentic application có deadline trước 7/12?,"Dự án Agentic application có task ""Fix lỗi crash app khi upload ảnh quá 5MB"" có deadline là 03/12 và task ""Tích hợp cổng thanh toán VNPay"" có deadline là 10/12. Do đó, chỉ có task ""Fix lỗi crash app khi upload ảnh quá 5MB"" có deadline trước ngày 7/12.",10,"✅ Accuracy: Phản hồi đã trả lời chính xác câu hỏi của người dùng bằng cách lọc và chỉ ra đúng task có deadline trước ngày 7/12. Câu trả lời cũng cung cấp thêm ngữ cảnh về một task khác không đáp ứng tiêu chí, giúp làm rõ lý do lựa chọn.✅ Tone: Giọng văn chuyên nghiệp, hữu ích và phù hợp với vai trò trợ lý. Ngôn ngữ tiếng Việt tự nhiên, không có lỗi dịch thuật.✅ Privacy/Fmt: Phản hồi không chứa bất kỳ ID kỹ thuật nào. Định dạng ngày (DD/MM) rõ ràng và dễ đọc. Mặc dù không dùng gạch đầu dòng, câu văn đơn giản và đủ rõ ràng cho một kết quả duy nhất."
